In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

In [2]:

vocab = ["The", "cat", "sat", "on", "the", "mat"]       # V = 6 token'lık sözlük
V, d_model = len(vocab), 4                               # her token 4 boyutlu vektör

class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(V, d_model)            # (V, d) tablo: id -> vektör
        self.head = nn.Linear(d_model, V, bias=False)    # (d -> V): vektör -> logitler
        # (gerçek modeldeki N decoder bloğu burada yok; sadece şekilleri görmek için)
    def forward(self, ids):                              # ids: (batch, seq)
        h = self.embed(ids)                              # (batch, seq, d_model)
        logits = self.head(h)                            # (batch, seq, V)
        return logits

model = TinyLM()
ids = torch.tensor([[0, 1, 2]])                          # "The cat sat" -> (1, 3)

for step in range(3):
    logits = model(ids)                                  # (1, seq, V)
    last = logits[:, -1, :]                              # SADECE son pozisyon: (1, V)
    probs = torch.softmax(last, dim=-1)                  # (1, V), toplam 1
    next_id = probs.argmax(dim=-1, keepdim=True)         # (1, 1)  (greedy)
    print(f"adım {step+1}: girdi {tuple(ids.shape)} -> logits {tuple(logits.shape)} "
          f"-> seçilen: {vocab[next_id.item()]}")
    ids = torch.cat([ids, next_id], dim=1)               # diziye ekle: seq 1 uzar

adım 1: girdi (1, 3) -> logits (1, 3, 6) -> seçilen: sat
adım 2: girdi (1, 4) -> logits (1, 4, 6) -> seçilen: sat
adım 3: girdi (1, 5) -> logits (1, 5, 6) -> seçilen: sat
